In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

from elasticsearch import Elasticsearch
import re

import sys
sys.path.append('Functions')
import preprocessing_fncs as ppf
import elastic_search_fncs as esf

In [2]:
# Details of the dataset
db_host = 'https://athena.london.gov.uk'
db_user = 'odbc_readonly'
db_pass = 'odbc_readonly'
db_port = '10099'
db_name = 'gla-ldd-external'

# Creates connection to the dataset
es = Elasticsearch(
    [f"{db_host}:{db_port}"],
    basic_auth=(db_user, db_pass),
    verify_certs=True,
    ca_certs='../Certificate/athena_es_full_chain.crt'
)

# Check connection
if es.ping():
    print("Connected to Elasticsearch!")
else:
    print("Could not connect to Elasticsearch.")

Connected to Elasticsearch!


In [3]:
# Define the list of years to query
years = list(range(2000, 2022))

# Create an empty list to store all individual DataFrames
all_dfs = []

# Loop through each year
for year in years:
    try:
        # Fetch the raw data for the given borough and year
        df = esf.get_residential_units(es=es, year=year)
        
        # Check if the data is empty or None
        if df is None or df.empty:
            print(f"Data retrieval failed: {year}, data is empty")
            continue

        print(f"Data retrieval successful: {year}")

        # Format the DataFrame using the custom function
        formatted_df = ppf.format_df(df)
        
        # Add metadata columns to track year
        formatted_df['year'] = year
        
        # Append the formatted DataFrame to the list
        all_dfs.append(formatted_df)

    except Exception as e:
        print(f"Data retrieval failed: {year}, error message: {e}")

# Concatenate all individual DataFrames into one
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    print("All data successfully merged")
else:
    print("No valid data retrieved")


Data retrieval successful: 2000
Data retrieval successful: 2001
Data retrieval successful: 2002
Data retrieval successful: 2003
Data retrieval successful: 2004
Data retrieval successful: 2005
Data retrieval successful: 2006
Data retrieval successful: 2007
Data retrieval successful: 2008
Data retrieval successful: 2009
Data retrieval successful: 2010
Data retrieval successful: 2011
Data retrieval successful: 2012
Data retrieval successful: 2013
Data retrieval successful: 2014
Data retrieval successful: 2015
Data retrieval successful: 2016
Data retrieval successful: 2017
Data retrieval successful: 2018
Data retrieval successful: 2019
Data retrieval successful: 2020
Data retrieval successful: 2021
All data successfully merged


In [4]:
final_df.to_csv("../Output/london_app_full_origin.csv", index=False)

In [5]:
final_df = pd.read_csv("../Output/london_app_full_origin.csv")

In [6]:
import ast

# Convert string to list
def parse_coords(value):
    if pd.notna(value):
        if isinstance(value, str):
            try:
                return ast.literal_eval(value)
            except Exception:
                return None
        return value

# Apply to both coordinate columns
final_df['wgs84_polygon.coordinates'] = final_df['wgs84_polygon.coordinates'].apply(parse_coords)
final_df['polygon.coordinates'] = final_df['polygon.coordinates'].apply(parse_coords)

In [ ]:
from shapely.geometry import shape
from shapely.errors import ShapelyError
from pyproj import Transformer

# Drop rows where both coordinate columns are NaN
final_df = final_df.dropna(subset=['wgs84_polygon.coordinates', 'polygon.coordinates'], how='all')

# Initialize transformer: EPSG:27700 (British National Grid) → EPSG:4326 (WGS84)
transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)

# Step 1: Generate a unified WGS84 coordinate column
def convert_coordinates(row):

    if row['wgs84_polygon.coordinates'] is not None:
        return row['wgs84_polygon.coordinates']
    
    else:

        coords = row['polygon.coordinates']
        
        try:
            return [
                [transformer.transform(x, y) for x, y in ring]
                for ring in coords
            ]
        except Exception:
            print('error')
            return None
    
    return None

final_df['wgs84_cleaned'] = final_df.apply(convert_coordinates, axis=1)

# Step 2: Compute centroid from cleaned WGS84 polygons
def compute_centroid(coords):
    try:
        if coords is not None and isinstance(coords, list) and len(coords) > 0:
            return shape({'type': 'Polygon', 'coordinates': coords}).centroid
    except ShapelyError:
        return None
    return None

final_df['geometry'] = final_df['wgs84_cleaned'].apply(compute_centroid)

# Step 3: Convert to GeoDataFrame in EPSG:4326
gdf_points = gpd.GeoDataFrame(final_df, geometry='geometry', crs='EPSG:4326')

# Step 4: Load LSOA boundaries and convert to WGS84
london_lsoa = gpd.read_file("../Output/Geo Output/London_LSOA_2021.shp")
london_lsoa = london_lsoa.to_crs('EPSG:4326')

# Step 5: Perform spatial join to assign LSOA codes
gdf_joined = gpd.sjoin(
    gdf_points,
    london_lsoa[['lsoa21cd', 'geometry']],
    how='left',
    predicate='within'
)

# Step 6: Add LSOA code to original DataFrame
final_df['lsoa21cd'] = gdf_joined['lsoa21cd']

final_df.dropna(subset=['lsoa21cd'])

# Step 7: Report results
print(f"Number of successfully matched points: {final_df['lsoa21cd'].notna().sum()}")

Number of successfully matched points: 122016


In [8]:
final_df.to_csv("../Output/london_app_full_converted.csv", index=False)

In [9]:
final_df

,uprn,pp_id,decision,total_no_proposed_residential_units,habitable_rooms_density,site_area,description,borough,street_name,site_name,...,wgs84_polygon.coordinates,wgs84_polygon.type,wgs84_polygon,polygon.coordinates,year,polygon,total_no_affordable_units,wgs84_cleaned,geometry,lsoa21cd
1,NaN,NaN,NaN,9,0.0,0.10,Redevelopment of public house to provide 9 flats.,Hounslow,Hounslow Road,Oxford Arms P.H.,...,"[[[-0.392094, 51.433757], [-0.392095, 51.43373...",Polygon,NaN,None,2000,NaN,NaN,"[[[-0.392094, 51.433757], [-0.392095, 51.43373...",POINT (-0.3921159999999991 51.433744),E01002608
2,NaN,NaN,NaN,2,0.0,0.12,Demoltion of existing house and erection of 2 ...,Hounslow,Jersey Road,251,...,"[[[-0.344967, 51.484196], [-0.344968, 51.48416...",Polygon,NaN,None,2000,NaN,NaN,"[[[-0.344967, 51.484196], [-0.344968, 51.48416...",POINT (-0.3449895 51.4841825),E01002680
3,100021571489,NaN,NaN,1,NaN,NaN,The erection of a 2 bedroomed bungalow.,Hounslow,Upper Sutton Lane,"Land to rear of, 77",...,"[[[-0.373077, 51.481873], [-0.373078, 51.48184...",Polygon,NaN,None,2000,NaN,NaN,"[[[-0.373077, 51.481873], [-0.373078, 51.48184...",POINT (-0.3730992459574458 51.4818600516312),E01002628
4,NaN,NaN,NaN,2,0.0,0.12,Erection of 2 x five-bedroom houses with dorme...,Hounslow,Jersey Road,251,...,"[[[-0.344967, 51.484196], [-0.344968, 51.48416...",Polygon,NaN,None,2000,NaN,NaN,"[[[-0.344967, 51.484196], [-0.344968, 51.48416...",POINT (-0.3449895 51.4841825),E01002680
5,NaN,NaN,NaN,2,NaN,NaN,Change of use of ground floor retail unit and ...,Hounslow,Hamilton Road,36A,...,"[[[-0.304882, 51.488155], [-0.304883, 51.48812...",Polygon,NaN,None,2000,NaN,NaN,"[[[-0.304882, 51.488155], [-0.304883, 51.48812...",POINT (-0.3049045 51.488141999999996),E01002567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124530,202221734,PP-09574487,Refused,8,NaN,NaN,Change of use of first and second floors offic...,Brent,NaN,"Ground, first and second floor offices, Moran ...",...,None,NaN,NaN,"[[[521857.7, 184781.35], [521852.4, 184791.5],...",2021,NaN,NaN,"[[(-0.24395448409224368, 51.54877433279962), (...",POINT (-0.2434823936232951 51.54850789705744),E01000638
124535,202206068,PP-09068640,Approved,7,NaN,NaN,Certificate of lawfulness for existing use as ...,Brent,NaN,"199 Brondesbury Park, Brondesbury, London, NW2...",...,None,NaN,NaN,"[[[522992.2, 184600.3], [522978.8, 184585.55],...",2021,NaN,NaN,"[[(-0.2276642681478578, 51.54690131880965), (-...",POINT (-0.2279109861174194 51.54677001691387),E01000488
124536,100023640606,PP-09239161,NaN,12,NaN,NaN,Erection of mansard roof over whole building t...,Croydon,NaN,Lavendar Apartments \r1A Mulgrave Road\rCroydo...,...,"[[[-0.095991, 51.369057], [-0.096279, 51.36903...",Polygon,NaN,None,2021,NaN,NaN,"[[[-0.095991, 51.369057], [-0.096279, 51.36903...",POINT (-0.0960749645169064 51.36889068016831),E01034149
124537,202210305,NaN,Approved,2,NaN,NaN,Prior approval for the construction of new dwe...,Brent,NaN,"Flats 1-8, Bhakti Court, 25 Tower Road, London...",...,None,NaN,NaN,"[[[522251.352, 184410.043], [522253.108, 18441...",2021,NaN,NaN,"[[(-0.23840898252872503, 51.54535227974506), (...",POINT (-0.2382511927431808 51.54533609304931),E01000643


In [10]:
print(final_df['lsoa21cd'].value_counts())

lsoa21cd
E01035716    324
E01004761    241
E01033708    226
E01004734    224
E01004736    209
            ... 
E01002536      1
E01001702      1
E01000465      1
E01000396      1
E01003189      1
Name: count, Length: 4955, dtype: int64


In [11]:
# Select the columns needed in the analysis
selected_columns = ['lsoa21cd', 'borough', 'year', 'description', 'decision', 'status', 'total_no_proposed_residential_units', 'wgs84_cleaned', 'geometry']
final_df_ = final_df[selected_columns]

In [12]:
final_df_

,lsoa21cd,borough,year,description,decision,status,total_no_proposed_residential_units,wgs84_cleaned,geometry
1,E01002608,Hounslow,2000,Redevelopment of public house to provide 9 flats.,NaN,Completed,9,"[[[-0.392094, 51.433757], [-0.392095, 51.43373...",POINT (-0.3921159999999991 51.433744)
2,E01002680,Hounslow,2000,Demoltion of existing house and erection of 2 ...,NaN,Superseded,2,"[[[-0.344967, 51.484196], [-0.344968, 51.48416...",POINT (-0.3449895 51.4841825)
3,E01002628,Hounslow,2000,The erection of a 2 bedroomed bungalow.,NaN,Lapsed,1,"[[[-0.373077, 51.481873], [-0.373078, 51.48184...",POINT (-0.3730992459574458 51.4818600516312)
4,E01002680,Hounslow,2000,Erection of 2 x five-bedroom houses with dorme...,NaN,Completed,2,"[[[-0.344967, 51.484196], [-0.344968, 51.48416...",POINT (-0.3449895 51.4841825)
5,E01002567,Hounslow,2000,Change of use of ground floor retail unit and ...,NaN,Completed,2,"[[[-0.304882, 51.488155], [-0.304883, 51.48812...",POINT (-0.3049045 51.488141999999996)
...,...,...,...,...,...,...,...,...,...
124530,E01000638,Brent,2021,Change of use of first and second floors offic...,Refused,Refused,8,"[[(-0.24395448409224368, 51.54877433279962), (...",POINT (-0.2434823936232951 51.54850789705744)
124535,E01000488,Brent,2021,Certificate of lawfulness for existing use as ...,Approved,Commenced,7,"[[(-0.2276642681478578, 51.54690131880965), (-...",POINT (-0.2279109861174194 51.54677001691387)
124536,E01034149,Croydon,2021,Erection of mansard roof over whole building t...,NaN,Completed,12,"[[[-0.095991, 51.369057], [-0.096279, 51.36903...",POINT (-0.0960749645169064 51.36889068016831)
124537,E01000643,Brent,2021,Prior approval for the construction of new dwe...,Approved,Completed,2,"[[(-0.23840898252872503, 51.54535227974506), (...",POINT (-0.2382511927431808 51.54533609304931)


In [13]:
final_df_.to_csv('../Output/london_app_selected_converted.csv', index=False)